In [1]:
# 简易的markdown转html工具。对markdown格式有额外要求，仅供自用
# 202408140124
'''
约定：.md文件中，全部使用html语法，除了：
- 标题可以用#，##，...
- 代码块可以用```...```，`...`
- 分隔线可以用------------，但连接符的个数必须>=6
- 数学公式可以用$...$, $$...$$
- 可以添加yaml头部，但必须以---在文件第一行开头，以----结尾（3个-开头，4个-结尾）。yaml头部的内容在转换为html时将被忽略

**尤其注意**：列表要用<ul><ol><li>等（嵌套列表转html逻辑过于复杂）；段落用<p></p>包围；加粗用<strong></strong>，斜体用<em></em>，删除线用<del></del>

总而言之，markdown尽量使用html编写
'''
def md_file_to_md_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        markdown_text = f.readlines()
    return markdown_text


def md_text_to_html_text(text):
    html_text=''
    code=0 # 0：当前不在代码块内；1：当前在代码块内
    pre_code=0 # 当前不在代码块内；1：当前在代码块内
    yaml_header=False # True：在yaml头部内；False：不在yaml头部内
    if text[0][0:3]=='---' and text[0][3]=='\n': # 如果第一行是yaml头部
        yaml_header=True

    
    for i,line in enumerate(text): # 遍历markdown文件的每一行

        if line.isspace(): # 如果是空行（仅由' ','\n','\t'组成）
            continue # 跳过空行

        elif line[0]=='#': # 判断标题
            if line[0:2]=='# ': # 如果是一级标题
                html_text+='<h1>'+line[2:-1]+'</h1>\n'
            elif line[0:3]=='## ': # 如果是二级标题
                html_text+='<h2>'+line[3:-1]+'</h2>\n'
            elif line[0:4]=='### ': # 如果是三级标题
                html_text+='<h3>'+line[4:-1]+'</h3>\n'
            elif line[0:5]=='#### ': # 如果是四级标题
                html_text+='<h4>'+line[5:-1]+'</h4>\n'
            elif line[0:6]=='##### ': # 如果是五级标题
                html_text+='<h5>'+line[6:-1]+'</h5>\n'
            elif line[0:7]=='###### ': # 如果是六级标题
                html_text+='<h6>'+line[7:-1]+'</h6>\n'

        elif line.lstrip()[0:3]=='```': # 判断代码块
            if pre_code==0: # 如果没有代码块，则开始一个新的代码块
                if line.lstrip()[3:5]=='py': # 如果是Python代码块
                    html_text+='<pre><code class="language-python">' # 不加'\n'
                elif line.lstrip()[3:5]=='js': # 如果是JavaScript代码块
                    html_text+='<pre><code class="language-javascript">' # 不加'\n'
                else: # 一般的的代码块
                    html_text+='<pre><code>'+'\n'
                pre_code=1
            elif pre_code==1: # 如果已经有代码块，则结束当前代码块
                html_text+='</code></pre>\n'
                pre_code=0

        elif yaml_header==True:
            if line[0:4]=='----' and line[4]=='\n':
                yaml_header=False # 结束yaml头部
                continue
            else:
                continue

        elif line[0:6]=='------': # 判断分隔线，分隔卡片
            html_text+='</div><div class="card">\n'

        else: # 其他情况，在每行内部判断
            l=''
            for j,char in enumerate(line): # 遍历每一行的每一个字符
                if char=='`':
                    if code==0:
                        l+='<code>'
                        code=1
                        continue
                    elif code==1:
                        l+='</code>'
                        code=0
                        continue
                l+=char
            html_text+=l
    return html_text

def generate_html_code_from_template(mycode='\n\n', prism=False, mathjax=False, css=[]): # html代码生成器。仅生成大致模板，具体细节可手动调节。
    htmlText=''
    htmlText += '<!DOCTYPE html>\n'
    htmlText += '<html>\n'
    htmlText += '<head>\n'
    htmlText += '<title>测试版个人网站</title>\n'
    htmlText += '<meta charset="utf-8"><!--字符集-->\n'
    htmlText += '<meta name="viewport" content="width=device-width, initial-scale=1.0"><!--页面视图-->\n'
    htmlText += '<meta name="keywords" content="celestopia, personal website, test"><!--页面关键词-->\n'
    htmlText += '<meta name="description" content="测试版个人网站"><!--页面描述-->\n'
    htmlText += '<meta name="author" content="Celestopia"><!--作者信息-->\n'
    htmlText += '<meta http-equiv="refresh" content="300"><!--自动刷新时间-->\n'
    htmlText += '<link rel="icon" href="/media/icons/icon_32x32.png"><!--网站图标-->\n'
    htmlText += '<link rel="stylesheet" type="text/css" href="/css/basic.css"><!--样式表-->\n'

    if css != None:
        for file in css:
            htmlText += f'<link rel="stylesheet" type="text/css" href="/css/{file}.css"><!--{file}样式表-->\n'

    if prism==True:
        htmlText += '<!--引入Prism.js实现语法高亮-->\n'
        htmlText += '<link href="https://cdn.jsdelivr.net/npm/prismjs@1/themes/prism.css" rel="stylesheet" /><!--引入Prism.js的CSS主题文件-->\n'
        htmlText += '<script src="https://cdn.jsdelivr.net/npm/prismjs@1/prism.min.js"></script><!--引入Prism.js的JavaScript文件，用于代码高亮-->\n'
        htmlText += '<script src="https://cdn.jsdelivr.net/npm/prismjs@1/components/prism-python.min.js"></script><!--引入Python语言插件-->\n'
        htmlText += "<script>document.addEventListener('DOMContentLoaded',(event) =>{Prism.highlightAll();});</script><!--确保在DOM完全加载后再执行Prism.js的高亮代码-->\n"

    if mathjax==True:
        htmlText += '<!--引入MathJax显示数学公式-->\n'
        htmlText += '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML"></script>\n'
        htmlText += '<script type="text/x-mathjax-config">'
        htmlText += "MathJax.Hub.Config({tex2jax:{inlineMath:[['$','$'],['\$','\$']]},});</script>\n"
        htmlText += '</head>\n'
    
    # --------------------------------
    
    htmlText += '<body>\n'
    htmlText += '  <div class="background" style="background-image:url(/media/backgrounds/background.jpg)"><!--修改图片路径以改变背景图-->\n'
    htmlText += '    <script type="text/javascript" src="/js/topnav_html_code.js"></script><!--导航栏-->\n'
    htmlText += '    <script type="text/javascript" src="/js/header_html_code.js"></script><!--头部-->\n'
    htmlText += '\n'
    htmlText += '    <div class="row">\n'
    htmlText += '\n'
    htmlText += '      <div class="left-sidebar" style="visibility: hidden;"><!--将visibility改为visible以启用左侧边栏-->\n'
    htmlText += '        <h2 style="text-align:center">相关链接</h2>\n'
    htmlText += '        <p>占位</p>\n'
    htmlText += '        <p>占位</p>\n'
    htmlText += '        <p>占位</p>\n'
    htmlText += '        <p>占位</p>\n'
    htmlText += '      </div>\n'
    htmlText += '\n'
    htmlText += '      <div class="left-sidebar" style="position:static">\n'
    htmlText += '        <!--占位用-->\n'
    htmlText += '      </div>\n'
    htmlText += '\n'
    htmlText += '      <div class="main">\n'
    htmlText += '        <div class="card">\n'
    htmlText += '          <!--以下为主体内容-->\n'
    htmlText += '\n'
    htmlText += '\n'
    htmlText += '\n'
    # --------------------------------

    htmlText += mycode
    
    # -------------------------------
    htmlText += '\n'
    htmlText += '\n'
    htmlText += '\n'
    htmlText += '        <!--以上为主体内容-->\n'
    htmlText += '          </div>\n'
    htmlText += '      </div>\n'
    htmlText += '\n'
    htmlText +='      <div class="right-sidebar" style="visibility: hidden;"><!--将visibility改为visible以启用右侧边栏-->\n'
    htmlText +='        <h2 style="text-align:center">右侧边栏</h2>\n'
    htmlText +='        <p>占位</p>\n'
    htmlText +='        <p>占位</p>\n'
    htmlText +='        <p>占位</p>\n'
    htmlText +='        <p>占位</p>\n'
    htmlText +='      </div>\n'
    htmlText += '\n'
    htmlText += '    <script type="text/javascript" src="/js/footer_html_code.js"></script><!--页脚-->\n'
    htmlText += '  </div>\n'
    htmlText += '</body>\n'
    htmlText += '</html>'
    print(htmlText)


def md2html(file_path, _prism=False, _mathjax=False, _css=[]):
    generate_html_code_from_template(md_text_to_html_text(md_file_to_md_text(file_path)), prism=_prism, mathjax=_mathjax, css=_css)



In [4]:
md2html(file_path="E:\\生活收藏\\知识随记\\20240813_json.md", _prism=True, _mathjax=True, _css=["article", "code"])

<!DOCTYPE html>
<html>
<head>
<title>测试版个人网站</title>
<meta charset="utf-8"><!--字符集-->
<meta name="viewport" content="width=device-width, initial-scale=1.0"><!--页面视图-->
<meta name="keywords" content="celestopia, profile, record"><!--页面关键词-->
<meta name="description" content="测试版个人网站"><!--页面描述-->
<meta name="author" content="Celestopia"><!--作者信息-->
<meta http-equiv="refresh" content="300"><!--自动刷新时间-->
<link rel="icon" href="/media/icons/icon_32x32.png"><!--网站图标-->
<link rel="stylesheet" type="text/css" href="/css/basic.css"><!--样式表-->
<link rel="stylesheet" type="text/css" href="/css/article.css"><!--article样式表-->
<link rel="stylesheet" type="text/css" href="/css/code.css"><!--code样式表-->
<!--引入Prism.js实现语法高亮-->
<link href="https://cdn.jsdelivr.net/npm/prismjs@1/themes/prism.css" rel="stylesheet" /><!--引入Prism.js的CSS主题文件-->
<script src="https://cdn.jsdelivr.net/npm/prismjs@1/prism.min.js"></script><!--引入Prism.js的JavaScript文件，用于代码高亮-->
<script src="https://cdn.jsdelivr.net/npm/prismjs@1/c